# Phase 4 — Power BI semantic model

**Global 2nd-Tier Metals Transparency & Cost-at-Risk Engine**

Every measure below sits on the gold layer produced by
`scripts/build_cost_at_risk.py` (`Mock_Project/data/gold/`). Nothing is
recomputed in DAX that the pipeline already proved in Python — the semantic
model presents the numbers, it does not invent them.

Two exceptions are deliberate, and both are marked where they appear:

- **HHI** is written dynamically so it responds to the metal and region
  slicers. The precomputed `fact_metal_concentration` table is kept as the
  reconciliation check.
- **CBAM certificate cost** is priced against a what-if parameter, because
  the carbon price is an assumption the business wants to move, not a fact
  the pipeline should freeze.

## 1. Relationships

All single-direction, many-to-one, fact → dimension.

| From | To | Note |
|---|---|---|
| `fact_should_cost[po_date]` | `dim_date[date]` | mark `dim_date` as the date table |
| `fact_metal_exposure[po_date]` | `dim_date[date]` | |
| `fact_should_cost[index_metal_id]` | `dim_metal[metal_id]` | the metal the clause indexes on |
| `fact_metal_exposure[metal_id]` | `dim_metal[metal_id]` | the metal physically embedded |
| `fact_cost_at_risk[index_metal_id]` | `dim_metal[metal_id]` | |
| `fact_tier2_allocation[metal_id]` | `dim_metal[metal_id]` | |
| `fact_ntier_coverage[metal_id]` | `dim_metal[metal_id]` | |
| `fact_tier2_allocation[t2_entity_id]` | `dim_supplier_tier2_risk[t2_entity_id]` | |
| `fact_*[tier1_id]` | `dim_supplier_tier1[tier1_id]` | exposure, should_cost, cost_at_risk, allocation, coverage, claims |
| `fact_should_cost[part_id]` | `dim_part[part_id]` | |
| `dim_supplier_tier2_risk[country_code]` | `dim_country_risk[country_code]` | |

`kpi_executive_summary` stays **disconnected**. It exists to reconcile the
report against the pipeline, not to be sliced.

## 2. Three traps in this model

**Never sum spend from `fact_metal_exposure`.** `net_value_eur` repeats on
every metal row — 44,876 rows carry 11,760 purchase orders. Spend comes from
`fact_should_cost` only. This is the single easiest way to publish a report
that is wrong by a factor of four.

**Never relate `dim_country_risk` to `dim_supplier_tier1`.** Country already
reaches Tier-1 through `fact_tier2_allocation`; adding the direct path closes
a loop and Power BI will deactivate one of the relationships silently.

**Never relate `dim_part` to `dim_supplier_tier1`** on `primary_tier1_id` —
same loop, same silent failure.

## 3. Parameter and helper tables

> Copy-paste source of truth: **`Mock_Project/powerbi/measures.dax`**. This
> notebook explains the measures; that file is the one to paste from.

These two are **calculated tables**, not files. They do not exist in
`data/gold/` — you create them inside Power BI with
**Modeling → New table**, *not* New measure.

**Create the table before the measure that references it.** A measure cannot
bring its own table into existence, so pasting `[Carbon Price Value]` first
gives *"cannot find table 'Carbon Price'"*.

`GENERATESERIES` always names its column `[Value]`. Referencing it as
`'Carbon Price'[Carbon Price]` fails. Keep `[Value]`, or rename the column in
Model view and update the measure to match.

```dax
Carbon Price = GENERATESERIES ( 0, 200, 5 )
```

```dax
Carbon Price Value =
SELECTEDVALUE ( 'Carbon Price'[Value], 85 )
```

*Easier alternative:* **Modeling → New parameter → Numeric range** builds the
table, the column and the slider measure in one step. It names the column
after the parameter, so adjust `[Carbon Price Value]` accordingly.

Leave `Carbon Price` **disconnected** — it is a parameter, not a dimension.
Do not relate it to anything.

The waterfall on page 1 needs a second disconnected table to order its
columns. Also **Modeling → New table**.

```dax
Bridge Steps =
DATATABLE (
    "Step", STRING,
    "Step Order", INTEGER,
    {
        { "Base price",          1 },
        { "Metal-justified",     2 },
        { "Inflation-justified", 3 },
        { "Unexplained",         4 },
        { "Actual spend",        5 }
    }
)
```

```dax
Bridge Value =
SWITCH (
    SELECTEDVALUE ( 'Bridge Steps'[Step Order] ),
    1, [Base Price Value],
    2, [Metal-Justified],
    3, [Inflation-Justified],
    4, [Unexplained],
    5, [Purchasing Spend]
)
```

`[Bridge Value]` references five measures from section 5, so create those
first or it will fail for the same reason.

### Data types to set before any of this works

| Column | Type |
|---|---|
| `dim_date[date]` | Date, then **Mark as date table** |
| `fact_should_cost[po_date]`, `fact_metal_exposure[po_date]` | Date |
| `fact_cost_at_risk[valid_to]` | Date |
| every `*_flag` column | True/False (Logical) |

The flags matter most. They arrive as the text `"True"`/`"False"` if type
detection is off, and every `= TRUE()` filter then silently returns blank
rather than erroring — so `[Unjustified Surcharge (Annualised)]` would read
zero and nobody would notice.


## 4. Spend and exposure

```dax
Purchasing Spend = SUM ( fact_should_cost[net_value_eur] )
```

```dax
Embedded Metal Value = SUM ( fact_metal_exposure[metal_spend_eur] )
```

```dax
Metal Share of Spend = DIVIDE ( [Embedded Metal Value], [Purchasing Spend] )
```

```dax
Embedded Metal (t) = DIVIDE ( SUM ( fact_metal_exposure[metal_kg] ), 1000 )
```

## 5. The should-cost bridge

Actual price against the contract clause replayed on the published index.
The residual is what a price letter has to defend.

```dax
Should-Cost = SUM ( fact_should_cost[should_cost_eur] )
```

```dax
Price Variance = [Purchasing Spend] - [Should-Cost]
```

```dax
Price Variance % = DIVIDE ( [Price Variance], [Should-Cost] )
```

```dax
Base Price Value =
SUMX (
    fact_should_cost,
    fact_should_cost[base_price_eur] * fact_should_cost[order_qty]
)
```

```dax
Metal-Justified = SUM ( fact_should_cost[metal_justified_eur] )
```

```dax
Inflation-Justified = SUM ( fact_should_cost[inflation_justified_eur] )
```

```dax
Unexplained = SUM ( fact_should_cost[unexplained_eur] )
```

```dax
Unexplained % = DIVIDE ( [Unexplained], [Should-Cost] )
```

The four components reconcile exactly to the invoice — the Python asserts it
on every run, so the waterfall cannot silently drift. Keep this measure on a
hidden diagnostics page.

```dax
Bridge Check =
VAR Reconstructed =
    [Base Price Value] + [Metal-Justified] + [Inflation-Justified] + [Unexplained]
RETURN
    IF ( ABS ( Reconstructed - [Purchasing Spend] ) < 1, "OK", "BROKEN" )
```

## 6. Cost-at-risk

Arithmetic on contracts already signed, not a forecast.

```dax
Cost at Risk (12m) = SUM ( fact_cost_at_risk[cost_at_risk_12m_eur] )
```

```dax
Clause Leakage = SUM ( fact_cost_at_risk[clause_leakage_eur] )
```

```dax
Reset Lag Cost = SUM ( fact_cost_at_risk[reset_lag_cost_eur] )
```

```dax
Repricing Exposure = SUM ( fact_cost_at_risk[repricing_exposure_eur] )
```

```dax
Unjustified Surcharge (Annualised) =
CALCULATE (
    SUM ( fact_surcharge_claims[annualised_unexplained_eur] ),
    fact_surcharge_claims[material_flag] = TRUE ()
)
```

```dax
Total Addressable =
[Unjustified Surcharge (Annualised)]
    + [Cost at Risk (12m)]
    + [Clause Leakage]
    + [Reset Lag Cost]
```

```dax
Addressable % of Spend = DIVIDE ( [Total Addressable], [Purchasing Spend] )
```

Measure names cannot usefully contain comparison operators — a name like
`Spend Expiring <=12m` breaks every reference to it. Spell the bound out.

```dax
Spend Expiring Within 12m =
CALCULATE (
    SUM ( fact_cost_at_risk[spend_12m_eur] ),
    fact_cost_at_risk[months_to_expiry] > 0,
    fact_cost_at_risk[months_to_expiry] <= 12
)
```

```dax
Contracts Expiring Within 6m =
CALCULATE (
    COUNTROWS ( fact_cost_at_risk ),
    fact_cost_at_risk[months_to_expiry] > 0,
    fact_cost_at_risk[months_to_expiry] <= 6
)
```


## 7. Calculated columns

On `fact_cost_at_risk`, for the expiry profile on page 1.

```dax
Expiry Bucket =
SWITCH (
    TRUE (),
    fact_cost_at_risk[months_to_expiry] <= 0,  "Expired",
    fact_cost_at_risk[months_to_expiry] <= 3,  "0-3 months",
    fact_cost_at_risk[months_to_expiry] <= 6,  "4-6 months",
    fact_cost_at_risk[months_to_expiry] <= 12, "7-12 months",
    fact_cost_at_risk[months_to_expiry] <= 24, "13-24 months",
    "24+ months"
)
```

```dax
Expiry Bucket Sort =
SWITCH (
    TRUE (),
    fact_cost_at_risk[months_to_expiry] <= 0,  0,
    fact_cost_at_risk[months_to_expiry] <= 3,  1,
    fact_cost_at_risk[months_to_expiry] <= 6,  2,
    fact_cost_at_risk[months_to_expiry] <= 12, 3,
    fact_cost_at_risk[months_to_expiry] <= 24, 4,
    5
)
```

Set `Expiry Bucket` to sort by `Expiry Bucket Sort`, otherwise the axis
comes out alphabetically and reads as nonsense. The sort column must read
`months_to_expiry`, not `[Expiry Bucket]` — a sort column derived from the
column it sorts is a circular dependency.


## 8. FX decomposition (S5)

The exchange quotes in USD; the P&L reports in EUR. When the dollar
strengthens during a rally the euro-denominated increase is larger than the
headline, and that difference lands entirely in the margin.

These compare the first and last month of the price book directly. An earlier
version used `LASTNONBLANK`, which returns the wrong month whenever the visual
carries no date context — a card with no date slicer would quietly show a
number from the wrong period. Both measures assume a single selected metal.

```dax
Index Move (EUR) =
VAR BaseMonth = MIN ( dim_price_book[month_index] )
VAR LastMonth = MAX ( dim_price_book[month_index] )
VAR BaseValue =
    CALCULATE ( SUM ( dim_price_book[eur_per_kg] ), dim_price_book[month_index] = BaseMonth )
VAR LastValue =
    CALCULATE ( SUM ( dim_price_book[eur_per_kg] ), dim_price_book[month_index] = LastMonth )
RETURN
    DIVIDE ( LastValue, BaseValue ) - 1
```

```dax
Index Move (Quote Ccy) =
VAR BaseMonth = MIN ( dim_price_book[month_index] )
VAR LastMonth = MAX ( dim_price_book[month_index] )
VAR BaseValue =
    CALCULATE ( SUM ( dim_price_book[settlement_price] ), dim_price_book[month_index] = BaseMonth )
VAR LastValue =
    CALCULATE ( SUM ( dim_price_book[settlement_price] ), dim_price_book[month_index] = LastMonth )
RETURN
    DIVIDE ( LastValue, BaseValue ) - 1
```

```dax
FX Amplification =
DIVIDE ( 1 + [Index Move (EUR)], 1 + [Index Move (Quote Ccy)] ) - 1
```

```dax
Metal Price (EUR/kg) =
VAR LastMonth = MAX ( dim_price_book[month_index] )
RETURN
    CALCULATE ( SUM ( dim_price_book[eur_per_kg] ), dim_price_book[month_index] = LastMonth )
```


## 9. N-tier transparency and concentration

Coverage is a share of **value**, not a count of suppliers. Ten declared
small sub-suppliers do not compensate for one undeclared large one.

```dax
N-Tier Coverage % =
DIVIDE (
    SUM ( fact_ntier_coverage[covered_spend_eur] ),
    SUM ( fact_ntier_coverage[tier1_metal_spend_eur] )
)
```

```dax
Blind Metal Spend = SUM ( fact_ntier_coverage[blind_spend_eur] )
```

```dax
Undeclared Pairs =
CALCULATE (
    COUNTROWS ( fact_ntier_coverage ),
    fact_ntier_coverage[is_covered] = FALSE ()
)
```

Declared shares are rounded to four decimals at source, so a Tier-1 can
declare 100.01% of its own sourcing. The pipeline now caps coverage at 1 and
floors blind spend at 0 — without that, this KPI rendered as 100.01% and blind
spend went negative. The over-declaration is kept as a flag rather than
silently discarded, because it is the honest answer to "how far do you trust
the declared shares?"

```dax
Over-Declared Pairs =
CALCULATE (
    COUNTROWS ( fact_ntier_coverage ),
    fact_ntier_coverage[over_declared_flag] = TRUE ()
)
```

```dax
Allocated Tier-2 Spend = SUM ( fact_tier2_allocation[allocated_spend_eur] )
```

```dax
Unverified Declared Spend =
CALCULATE (
    [Allocated Tier-2 Spend],
    fact_tier2_allocation[unverified_flag] = TRUE ()
)
```

```dax
Unverified Share =
DIVIDE (
    [Unverified Declared Spend],
    CALCULATE ( [Allocated Tier-2 Spend], ALL ( fact_tier2_allocation[unverified_flag] ) )
)
```


### Herfindahl-Hirschman Index — written dynamically

Above 2,500 counts as highly concentrated. Computing it on the raw declared
**names** versus on the **resolved entities** shows exactly how much risk the
aliasing was hiding — and because these are measures rather than the
precomputed column, the comparison survives every slicer.

Both iterate `fact_tier2_allocation` rather than the Tier-2 dimension, so the
two measures sit on the same table and the comparison cannot be skewed by a
relationship filtering one side but not the other.

```dax
HHI (Resolved) =
VAR Total = [Allocated Tier-2 Spend]
RETURN
    IF (
        Total > 0,
        SUMX (
            VALUES ( fact_tier2_allocation[t2_entity_id] ),
            VAR EntityShare = DIVIDE ( [Allocated Tier-2 Spend], Total )
            RETURN EntityShare * EntityShare
        ) * 10000
    )
```

```dax
HHI (As Declared) =
VAR Total = [Allocated Tier-2 Spend]
RETURN
    IF (
        Total > 0,
        SUMX (
            VALUES ( fact_tier2_allocation[declared_supplier_name] ),
            VAR NameShare = DIVIDE ( [Allocated Tier-2 Spend], Total )
            RETURN NameShare * NameShare
        ) * 10000
    )
```

```dax
HHI Understatement = [HHI (Resolved)] - [HHI (As Declared)]
```

```dax
Concentration Verdict =
SWITCH (
    TRUE (),
    ISBLANK ( [HHI (Resolved)] ), "No allocated spend",
    [HHI (Resolved)] >= 2500,     "Highly concentrated",
    [HHI (Resolved)] >= 1500,     "Moderately concentrated",
    "Competitive"
)
```

```dax
Largest Entity Share =
VAR Total = [Allocated Tier-2 Spend]
RETURN
    DIVIDE (
        MAXX ( VALUES ( fact_tier2_allocation[t2_entity_id] ), [Allocated Tier-2 Spend] ),
        Total
    )
```


## 10. Tier-2 risk

Cyber maturity, geopolitics and compliance are only meaningful next to
exposure. A weak supplier you barely buy from is a nuisance; a weak supplier
carrying a fifth of your metal value is business continuity.

```dax
Spend-Weighted Risk = SUM ( dim_supplier_tier2_risk[spend_weighted_risk_eur] )
```

```dax
Composite Risk (Spend-Weighted) =
DIVIDE (
    SUMX (
        dim_supplier_tier2_risk,
        dim_supplier_tier2_risk[composite_risk_score]
            * dim_supplier_tier2_risk[allocated_spend_eur]
    ),
    SUM ( dim_supplier_tier2_risk[allocated_spend_eur] )
)
```

```dax
Critical Entities =
CALCULATE (
    COUNTROWS ( dim_supplier_tier2_risk ),
    dim_supplier_tier2_risk[risk_tier] = "Critical"
)
```

```dax
Metal Value Below Cyber Line =
CALCULATE (
    SUM ( dim_supplier_tier2_risk[allocated_spend_eur] ),
    dim_supplier_tier2_risk[cyber_maturity_score] < 50
)
```

```dax
Cyber-Exposed Share =
DIVIDE (
    [Metal Value Below Cyber Line],
    CALCULATE (
        SUM ( dim_supplier_tier2_risk[allocated_spend_eur] ),
        ALL ( dim_supplier_tier2_risk )
    )
)
```

```dax
Hidden Single-Source Entities =
CALCULATE (
    COUNTROWS ( dim_supplier_tier2_risk ),
    dim_supplier_tier2_risk[hidden_by_aliasing_flag] = TRUE ()
)
```

```dax
Max Tier-1 Behind One Entity = MAX ( dim_supplier_tier2_risk[n_tier1_dependent] )
```

## 11. CBAM

A carbon liability is owed on **mass**, not on money, so these measures run
off `allocated_metal_kg` rather than allocated spend. Only imports carry a
certificate — intra-EU production is already inside the ETS and must not be
double counted, which `cbam_liable_flag` enforces in the pipeline.

```dax
CBAM Liable Metal (t) =
DIVIDE (
    CALCULATE (
        SUM ( fact_tier2_allocation[allocated_metal_kg] ),
        fact_tier2_allocation[cbam_liable_flag] = TRUE ()
    ),
    1000
)
```

```dax
CBAM Embedded Emissions (tCO2e) =
DIVIDE (
    SUMX (
        FILTER (
            fact_tier2_allocation,
            fact_tier2_allocation[cbam_liable_flag] = TRUE ()
        ),
        fact_tier2_allocation[allocated_metal_kg]
            * COALESCE ( RELATED ( dim_supplier_tier2_risk[co2e_kg_per_kg_metal] ), 0 )
    ),
    1000
)
```

```dax
CBAM Certificate Cost = [CBAM Embedded Emissions (tCO2e)] * [Carbon Price Value]
```

```dax
Imported Share of Metal Value =
DIVIDE (
    CALCULATE ( [Allocated Tier-2 Spend], fact_tier2_allocation[import_flag] = TRUE () ),
    CALCULATE ( [Allocated Tier-2 Spend], ALL ( fact_tier2_allocation[import_flag] ) )
)
```

The switching value — what the same tonnage would cost at the cleanest
intensity available in the resolved network — is what turns a compliance page
into a sourcing recommendation.

```dax
CBAM Cost at Benchmark Intensity =
VAR Benchmark =
    MINX (
        FILTER (
            ALL ( dim_supplier_tier2_risk ),
            dim_supplier_tier2_risk[co2e_kg_per_kg_metal] > 0
        ),
        dim_supplier_tier2_risk[co2e_kg_per_kg_metal]
    )
VAR LiableKg =
    CALCULATE (
        SUM ( fact_tier2_allocation[allocated_metal_kg] ),
        fact_tier2_allocation[cbam_liable_flag] = TRUE ()
    )
RETURN
    DIVIDE ( LiableKg * Benchmark, 1000 ) * [Carbon Price Value]
```

```dax
CBAM Avoidable Cost = [CBAM Certificate Cost] - [CBAM Cost at Benchmark Intensity]
```

**Caveat to keep ready:** `allocated_metal_kg` inherits the declared shares,
which are self-reported and largely unaudited — `[Unverified Declared Spend]`
quantifies how much. This is a defensible planning estimate, not a customs
declaration.

## 12. Page 1 — Executive & Cost-at-Risk

1280 x 720, 12-column grid, 16 px gutters.

```
+--------------------------------------------------------------------------+
|  Global 2nd-Tier Metals - Cost-at-Risk     [Date] [Metal] [Tier-1]       |  y 0-72
+------------+------------+------------+------------+----------------------+
| Purchasing | Embedded   | TOTAL      | Unexplained| N-Tier Coverage      |  y 72-188
| Spend      | Metal Value| ADDRESSABLE| Variance % |                      |
+------------+------------+------------+------------+----------------------+
|                                     |                                    |
|  SHOULD-COST BRIDGE (waterfall)     |  COST-AT-RISK BY EXPIRY BUCKET     |  y 188-460
|  Base > Metal > Inflation >         |  (stacked column)                  |
|  Unexplained > Actual               |                                    |
+-------------------------------------+------------------------------------+
|  SURCHARGE CLAIMS TESTED (table)    |  CONTRACT ACTION LIST (table)      |  y 460-700
+-------------------------------------+------------------------------------+
```

| Visual | Type | Fields | Notes |
|---|---|---|---|
| Header slicers | Slicer x 3 | `dim_date[year_month]`, `dim_metal[metal_id]`, `dim_supplier_tier1[tier1_name]` | horizontal, Select all off |
| KPI 1 | Card | `[Purchasing Spend]` | EUR, millions, 1 dp |
| KPI 2 | Card | `[Embedded Metal Value]` | subtitle `[Metal Share of Spend]` |
| KPI 3 | Card | `[Total Addressable]` | accent colour, this is the headline |
| KPI 4 | Card | `[Unexplained %]` | conditional red above 0.5% |
| KPI 5 | KPI card | `[N-Tier Coverage %]`, trend `dim_date[year_month]`, target 0.85 | |
| Bridge | Waterfall | Category `'Bridge Steps'[Step]`, Y `[Bridge Value]` | breakdown off |
| Expiry | Stacked column | Axis `fact_cost_at_risk[Expiry Bucket]`; Y `[Cost at Risk (12m)]`, `[Clause Leakage]`, `[Reset Lag Cost]` | |
| Claims | Table | `tier1_name`, `claimed_surcharge_pct`, `justified_by_index_pct`, `justified_by_inflation_pct`, `unexplained_pct`, `annualised_unexplained_eur`, `justification_text` | filter `material_flag = True`, data bars on the last numeric |
| Actions | Table | `contract_id`, `index_metal_id`, `months_to_expiry`, `[Cost at Risk (12m)]`, `risk_driver`, `mitigation` | Top 15 by `total_cost_at_risk_eur` |

## 13. Page 2 — N-Tier Resilience & CBAM

```
+--------------------------------------------------------------------------+
|  N-Tier Resilience & Carbon Exposure  [Metal] [Region] [Carbon EUR/t <->] |  y 0-72
+------------+------------+------------+------------+----------------------+
| Coverage   | Blind Spend| HHI        | Hidden     | CBAM Certificate     |  y 72-188
|            |            | declared > | Single-    | Cost                 |
|            |            | resolved   | Source     |                      |
+------------+------------+------------+------------+----------------------+
|                                     |                                    |
|  TIER-1 > TIER-2 FLOW (Sankey)      |  RISK vs EXPOSURE (scatter)        |  y 188-470
|                                     |  x spend, y cyber, size kg         |
+-------------------------------------+------------------------------------+
|  COVERAGE BY TIER-1 (bar)           |  CBAM BY ENTITY (table + bars)     |  y 470-700
+-------------------------------------+------------------------------------+
```

| Visual | Type | Fields | Notes |
|---|---|---|---|
| Carbon slider | Slicer | `'Carbon Price'[Carbon Price]` | single-value slider, default 85 |
| KPI 1 | Card | `[N-Tier Coverage %]` | target 85% |
| KPI 2 | Card | `[Blind Metal Spend]` | subtitle: undeclared Tier-1 x metal pairs |
| KPI 3 | Multi-row card | `[HHI (As Declared)]`, `[HHI (Resolved)]`, `[HHI Understatement]` | the before/after is the point |
| KPI 4 | Card | `[Hidden Single-Source Entities]` | subtitle `[Max Tier-1 Behind One Entity]` |
| KPI 5 | Card | `[CBAM Certificate Cost]` | title reacts to the slider |
| Sankey | Sankey (AppSource) | Source `dim_supplier_tier1[tier1_name]`, Destination `dim_supplier_tier2_risk[golden_name]`, Weight `[Allocated Tier-2 Spend]` | the visual that sells the project |
| Scatter | Scatter | X `[Allocated Tier-2 Spend]`, Y `cyber_maturity_score`, Size `allocated_metal_kg`, Legend `risk_tier`, Details `golden_name` | constant line at Y = 50, red |
| Coverage | Bar | Axis `dim_supplier_tier1[tier1_name]`, Value `[N-Tier Coverage %]` | red below 0.6 |
| CBAM | Table | `golden_name`, `country_code`, `[CBAM Liable Metal (t)]`, `co2e_kg_per_kg_metal`, `[CBAM Embedded Emissions (tCO2e)]`, `[CBAM Certificate Cost]`, `[CBAM Avoidable Cost]` | filter `import_flag = True` |

**Set this interaction deliberately:** the Sankey must **not** cross-filter the
scatter. Selecting a Tier-1 in a flow diagram and watching the scatter
collapse to a single dot destroys the comparison the scatter exists to make.
Format > Edit interactions > set the scatter to None.

**Drill-through:** define `dim_supplier_tier2_risk[golden_name]` as the
drill-through field on a hidden third page carrying the entity's full
compliance and disruption record. Right-clicking the Yunnan node then goes
straight to the evidence, which is the next question an interviewer asks.

## 14. Contract test

The measures above name roughly seventy columns across nine gold tables. If
the pipeline is refactored and a column is renamed, the report fails at
refresh time with an unhelpful error. The cell below fails here instead, in
one second, naming the column.

Run it after every pipeline change.

In [ ]:
from pathlib import Path

import pandas as pd

GOLD = Path.cwd().parents[0] / "data" / "gold"
if not GOLD.exists():                      # tolerate being run from the repo root
    GOLD = Path.cwd() / "Mock_Project" / "data" / "gold"

# Every column the DAX in this notebook and in powerbi/measures.dax depends on.
CONTRACT = {
    "fact_should_cost": [
        "po_date", "part_id", "tier1_id", "index_metal_id", "order_qty",
        "net_value_eur", "base_price_eur", "should_cost_eur",
        "metal_justified_eur", "inflation_justified_eur", "unexplained_eur",
    ],
    "fact_metal_exposure": [
        "po_date", "tier1_id", "metal_id", "metal_kg", "metal_spend_eur",
    ],
    "fact_cost_at_risk": [
        "contract_id", "tier1_id", "index_metal_id", "months_to_expiry",
        "valid_to", "spend_12m_eur", "cost_at_risk_12m_eur", "clause_leakage_eur",
        "reset_lag_cost_eur", "repricing_exposure_eur", "total_cost_at_risk_eur",
        "risk_driver", "mitigation",
    ],
    "fact_surcharge_claims": [
        "tier1_id", "tier1_name", "claimed_surcharge_pct", "justified_by_index_pct",
        "justified_by_inflation_pct", "unexplained_pct",
        "annualised_unexplained_eur", "justification_text", "material_flag",
    ],
    "fact_ntier_coverage": [
        "tier1_id", "metal_id", "tier1_metal_spend_eur", "covered_spend_eur",
        "blind_spend_eur", "is_covered", "over_declared_flag", "coverage_ratio",
    ],
    "fact_tier2_allocation": [
        "tier1_id", "metal_id", "t2_entity_id", "declared_supplier_name",
        "allocated_spend_eur", "allocated_metal_kg", "cbam_liable_flag",
        "import_flag", "unverified_flag",
    ],
    "dim_supplier_tier2_risk": [
        "t2_entity_id", "golden_name", "country_code", "composite_risk_score",
        "risk_tier", "allocated_spend_eur", "allocated_metal_kg",
        "spend_weighted_risk_eur", "cyber_maturity_score", "audit_overdue_flag",
        "hidden_by_aliasing_flag", "n_tier1_dependent", "co2e_kg_per_kg_metal",
    ],
    "dim_price_book": [
        "metal_id", "month_index", "eur_per_kg", "settlement_price",
        "index_move_vs_base", "idx_move_quote_ccy",
    ],
    "fact_metal_concentration": [
        "metal_id", "hhi_as_declared", "hhi_after_resolution",
    ],
}

missing = []
for table, columns in CONTRACT.items():
    path = GOLD / f"{table}.csv"
    if not path.exists():
        missing.append(f"{table}  <- table not found")
        continue
    have = set(pd.read_csv(path, nrows=1).columns)
    for column in columns:
        if column not in have:
            missing.append(f"{table}[{column}]")

print(f"gold layer : {GOLD}")
print(f"tables     : {len(CONTRACT)}")
print(f"columns    : {sum(len(c) for c in CONTRACT.values())}")
if missing:
    print("\nMISSING:")
    for m in missing:
        print(f"  {m}")
else:
    print("\nAll columns referenced by the DAX are present.")

assert not missing, f"{len(missing)} reference(s) broken -- the report will fail at refresh"

# Bounds the DAX silently relies on. A coverage ratio above 1 would put a
# 100.01% KPI on the executive page; negative blind spend would break the
# stacked bar on page 2.
cov = pd.read_csv(GOLD / "fact_ntier_coverage.csv")
assert cov["coverage_ratio"].between(0, 1).all(), "coverage_ratio outside 0..1"
assert cov["blind_spend_eur"].ge(0).all(), "negative blind_spend_eur"
print(f"coverage bounded 0..1, blind spend non-negative "
      f"({int(cov['over_declared_flag'].sum())} over-declared pair(s) flagged)")


### Reconciliation

The dynamic HHI measures must agree with the values the pipeline computed in
Python. If they diverge, either the DAX filter context is wrong or the
allocation changed shape — both are worth knowing before a demo.

In [ ]:
conc = pd.read_csv(GOLD / "fact_metal_concentration.csv")
alloc = pd.read_csv(GOLD / "fact_tier2_allocation.csv")


def hhi(series):
    total = series.sum()
    return float(((series / total) ** 2).sum() * 10_000) if total > 0 else 0.0


rows = []
for metal, g in alloc.groupby("metal_id"):
    expected = conc.loc[conc.metal_id == metal].iloc[0]
    rows.append({
        "metal_id": metal,
        "declared_dax_equivalent": hhi(g.groupby("declared_supplier_name").allocated_spend_eur.sum()),
        "declared_pipeline": expected.hhi_as_declared,
        "resolved_dax_equivalent": hhi(g.groupby("t2_entity_id").allocated_spend_eur.sum()),
        "resolved_pipeline": expected.hhi_after_resolution,
    })

check = pd.DataFrame(rows)
check["ok"] = (
    (check.declared_dax_equivalent - check.declared_pipeline).abs() < 1
) & (
    (check.resolved_dax_equivalent - check.resolved_pipeline).abs() < 1
)
print(check.to_string(index=False))
assert check.ok.all(), "dynamic HHI does not reconcile to the pipeline"
print("\nHHI measures reconcile.")